# 10 · Task basics: gliders, pilot ratings and a baseline without weather

Three pieces the task model needs, all from the official results (notebook 04):

| Step | Writes | What |
|---|---|---|
| Glider classes | `result_gliders`, `gliders` | one key per glider model (the same wing is written many ways) and its class: CCC, EN-D, EN-C, EN-B |
| Pilot ratings | `ratings`, `pilot_ratings`, `task_fields` | an Elo rating per pilot, recorded before every task, and the strength of each task's field |
| Baseline | `models/task_v0/` | the winner's speed and time and the share in goal, from route, timing, season, field and site history, with no weather: the bar the weather has to beat |

A few minutes on a normal runtime.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "results.parquet")), "Build the catalog tables first: database/results.parquet is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Glider classes

In [ ]:
run(f"{REPO}/scripts/gliders.py", DB)

## 4. Pilot ratings

The check line says how often the pilot rated higher before a task placed better (50% would be chance).

In [ ]:
run(f"{REPO}/scripts/pilot_ratings.py", DB)

## 5. Baseline: winner's time and goal share without weather

Scored on competitions left out (sites known) and on sites left out (a new site), next to two simple rules: the median of all tasks, and the site's own history.

In [ ]:
run(f"{REPO}/scripts/task_baseline.py", DB)

## 6. A look

In [ ]:
import matplotlib.pyplot as plt
p = pd.read_parquet(os.path.join(DB, "models", "task_v0", "predictions.parquet")).dropna(subset=["winner_h", "pred_winner_kmh_event_key"])
pred_h = p["ss_km"] / p["pred_winner_kmh_event_key"]
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(p["winner_h"], pred_h, s=8, color="#2a78d6", alpha=0.5)
lim = [0, max(p["winner_h"].max(), pred_h.max())]
ax.plot(lim, lim, color="#c3c2b7", lw=1)
ax.set(xlabel="winner's time (h)", ylabel="predicted, competitions left out (h)", title="Winner's time without weather")
ax.spines[["top", "right"]].set_visible(False)
plt.show()